# E02 / E03 — LightGBM 교체

## 목적

베이스라인(RandomForest)에서 **한 번에 하나씩** 바꾸며 각 변경의 기여도를 분리한다.

| 실험 | 모델 | 결측 처리 | 목적함수 | 무엇을 알 수 있나 |
|---|---|---|---|---|
| A | RandomForest | 중앙값 대치 | Gini | 기준선 (E01) |
| B0 | LightGBM 분류 | 중앙값 대치 | logloss | **A→B0 = 모델 교체 효과** |
| B | LightGBM 분류 | native | logloss | **B0→B = 결측 처리 효과** |
| C | LightGBM 회귀 | native | L2 = Brier | **B→C = 목적함수 효과** |

> 첫 실행 때는 B0 없이 A→B를 한 번에 비교했다.
> 모델 교체와 결측 처리가 동시에 바뀌어 기여도를 나눌 수 없었으므로 B0를 추가했다.
> (AGENTS.md §15 "한 번에 한 가지만 변경한다")

## 왜 이 변경들인가

**1. RandomForest는 방향이 맞지 않는다**

베이스라인은 `max_depth=10`으로 트리를 얕게 제한해 **과소적합(high bias)** 상태다.
그런데 RandomForest(Bagging)는 **variance만 줄이고 bias는 그대로 둔다.**
bias를 줄이려면 **Boosting**이 필요하다. (STRATEGY.md 난제 2)

**2. 학습 목적함수가 평가 지표와 다르다**

LightGBM 기본값은 logloss인데 대회 평가는 Brier다.
0/1을 L2 회귀로 학습하면 **MSE = Brier**가 되어 평가 지표를 직접 최소화한다.
(STRATEGY.md 난제 3)

**3. 결측이 정보다**

`asof_*` 결측 = "데뷔 첫 투구 / 첫 등판". 중앙값으로 채우면 이 정보가 사라진다.
LightGBM은 결측을 그대로 두고 별도 분기로 학습할 수 있다.

## 재현성

- `LGBM_VERSION` 으로 lightgbm 버전을 고정한다 (제출 requirements.txt와 동일)
- 실행 결과는 `results/e02_metrics.json` 에 저장된다 (노트북 출력이 사라져도 확인 가능)
- 환경 정보는 모델 artifact 에도 함께 기록되어 제출 시 버전 자동 통일에 쓰인다

## 검증 방식

```
학습: 2019~2023  (1,221,585행)
검증: 2024       (  253,507행)
```
시간 순서 홀드아웃. AGENTS.md §8 준수.

⚠️ 단일 홀드아웃이므로 소폭 차이(수 점 이내)는 판정할 수 없다.
시즌 롤링 검증 도입 후 재측정이 필요하다.

## 1. 준비

LightGBM이 없으면 설치한다. 아래 셀을 한 번만 실행하면 된다.

In [ ]:
# LightGBM 설치 (버전 고정)
# 제출 requirements.txt 와 반드시 같은 버전을 쓴다.
LGBM_VERSION = "4.5.0"

import sys, subprocess

def _ensure_lgbm():
    try:
        import lightgbm
    except ImportError:
        print(f"lightgbm=={LGBM_VERSION} 설치 중...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               f"lightgbm=={LGBM_VERSION}"])
        raise RuntimeError(
            "설치 완료. 커널을 재시작(Restart)한 뒤 처음부터 다시 실행하세요.")

    if lightgbm.__version__ != LGBM_VERSION:
        print(f"버전 불일치: 메모리에 로드됨 {lightgbm.__version__} != 요구 {LGBM_VERSION}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               f"lightgbm=={LGBM_VERSION}"])
        raise RuntimeError(
            f"lightgbm=={LGBM_VERSION} 설치 완료.\n"
            "이미 로드된 이전 버전이 메모리에 남아 있으므로 그대로 진행하면 안 됩니다.\n"
            "→ 커널을 재시작(Restart)한 뒤 처음부터 다시 실행하세요.")

    print(f"lightgbm {lightgbm.__version__} - 버전 일치")

_ensure_lgbm()

In [ ]:
import os
import time
import json
import platform
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

DATA_DIR    = "../../data"
MODEL_DIR   = "../../models"
RESULTS_DIR = "../../results"          # 실행 결과(metrics) 저장 위치

ID       = "row_id"
TARGET   = "control_success"
CAT_COLS = ["top_bottom", "game_type", "base_state"]

VALID_SEASON = 2024          # 검증에 사용할 시즌
SEED         = 42

# A(RandomForest)를 다시 학습할지. E01에서 이미 415.57을 확인했으므로 기본은 건너뛴다.
RUN_RF   = False
RF_SCORE = 415.57            # E01 기록값
RF_BRIER = 0.248769

# 재현에 필요한 환경 정보 (metrics 파일과 모델 artifact 에 함께 저장)
ENV = {
    "python":      platform.python_version(),
    "platform":    platform.platform(),
    "lightgbm":    lgb.__version__,
    "scikit-learn": sklearn.__version__,
    "pandas":      pd.__version__,
    "numpy":       np.__version__,
    "seed":        SEED,
}
print(json.dumps(ENV, indent=2, ensure_ascii=False))

## 2. 데이터 로딩

E01과 동일하다. `test.csv`에서 컬럼 목록을 가져와 47개 피처를 확정한다.

In [ ]:
t0 = time.time()

test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),
                        encoding="utf-8-sig", nrows=0).columns
FEATURES = [c for c in test_cols if c != ID]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"),
                    encoding="utf-8-sig", usecols=FEATURES + [TARGET])

print(f"로딩 완료 :: {time.time() - t0:.1f}s")
print("train:", train.shape, "| 피처:", len(FEATURES),
      f"(범주형 {len(CAT_COLS)}, 수치형 {len(NUM_COLS)})")
print("시즌:", train["season"].min(), "~", train["season"].max())
print(f"제구 성공률: {train[TARGET].mean():.4f}")
print()
print("시즌별 성공률")
print(train.groupby("season")[TARGET].agg(["size", "mean"]).round(4))

In [ ]:
# 시간 순서 분할 (랜덤 분할 금지 — AGENTS.md §8)
is_val = train["season"] == VALID_SEASON

X_tr_raw, y_tr = train.loc[~is_val, FEATURES], train.loc[~is_val, TARGET]
X_va_raw, y_va = train.loc[ is_val, FEATURES], train.loc[ is_val, TARGET]

print(f"학습 : {len(X_tr_raw):>9,}행  (2019~{VALID_SEASON-1})  성공률 {y_tr.mean():.4f}")
print(f"검증 : {len(X_va_raw):>9,}행  ({VALID_SEASON})        성공률 {y_va.mean():.4f}")

## 3. 공통 평가 함수

대회 공식 산식 그대로다.

```
Brier = mean((p - y)^2)
Score = max(0, 100000 * (1 - Brier / (r*(1-r))))
```

**학습 점수와 검증 점수를 함께 출력**한다.
두 값의 차이로 과적합/과소적합을 진단할 수 있다.

```
Train ≈ Valid   → 정상 (또는 과소적합)
Train >> Valid  → 과적합
```

In [ ]:
RESULTS = {}   # 실험 결과 모음

def brier_score(pred, y):
    return float(np.mean((np.asarray(pred) - np.asarray(y)) ** 2))

def comp_score(pred, y):
    """대회 산식. r은 해당 평가 데이터의 실제 평균."""
    y = np.asarray(y)
    r = y.mean()
    return max(0.0, 100000.0 * (1.0 - brier_score(pred, y) / (r * (1.0 - r))))

def report(name, pred_tr, pred_va, fit_sec=None, extra=None):
    b_tr, s_tr = brier_score(pred_tr, y_tr), comp_score(pred_tr, y_tr)
    b_va, s_va = brier_score(pred_va, y_va), comp_score(pred_va, y_va)

    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
    print(f"  Train  Brier {b_tr:.6f}   Score {s_tr:8.2f}")
    print(f"  Valid  Brier {b_va:.6f}   Score {s_va:8.2f}")
    print(f"  차이                        {s_tr - s_va:8.2f}")
    if fit_sec is not None:
        print(f"  학습 시간 {fit_sec:.1f}s")
    print(f"  예측 분포  min {pred_va.min():.4f}  "
          f"mean {pred_va.mean():.4f}  max {pred_va.max():.4f}  "
          f"std {pred_va.std():.4f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k}: {v}")

    RESULTS[name] = dict(train_brier=b_tr, train_score=s_tr,
                         valid_brier=b_va, valid_score=s_va,
                         fit_sec=fit_sec, extra=extra or {},
                         pred_valid_mean=float(pred_va.mean()),
                         pred_valid_std=float(pred_va.std()))
    return s_va


def save_metrics(filename="e02_metrics.json"):
    """실행 결과를 파일로 남긴다.

    노트북 출력은 커밋 시 사라질 수 있으므로, 숫자는 별도 파일로 보존한다.
    (리뷰어가 브랜치만 받아도 결과를 확인할 수 있게 하기 위함)
    """
    os.makedirs(RESULTS_DIR, exist_ok=True)
    payload = {
        "experiment":   "E02 / E03",
        "notebook":     "notebooks/exp/02_lgbm.ipynb",
        "run_at":       datetime.now().isoformat(timespec="seconds"),
        "environment":  ENV,
        "validation": {
            "scheme":      "time-based holdout",
            "train_seasons": f"2019~{VALID_SEASON - 1}",
            "valid_season":  VALID_SEASON,
            "n_train":     int(len(X_tr_raw)),
            "n_valid":     int(len(X_va_raw)),
            "train_rate":  float(y_tr.mean()),
            "valid_rate":  float(y_va.mean()),
        },
        "params":       PARAMS if "PARAMS" in globals() else None,
        "results":      RESULTS,
    }
    path = os.path.join(RESULTS_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"저장 완료: {path}")
    return path

## 4. A — RandomForest (기준선)

E01과 동일한 설정. 이미 **415.57**을 확인했으므로 기본적으로 건너뛴다.
다시 확인하고 싶으면 위에서 `RUN_RF = True`로 바꾼다. (약 3분 소요)

In [ ]:
if RUN_RF:
    pre_rf = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                               unknown_value=-1), CAT_COLS),
        ("num", SimpleImputer(strategy="median"), NUM_COLS),
    ])
    rf = Pipeline([
        ("pre", pre_rf),
        ("clf", RandomForestClassifier(n_estimators=100, max_depth=10,
                                       min_samples_leaf=200,
                                       n_jobs=-1, random_state=SEED)),
    ])
    t = time.time()
    rf.fit(X_tr_raw, y_tr)
    sec = time.time() - t

    report("A. RandomForest + 중앙값 대치",
           rf.predict_proba(X_tr_raw)[:, 1],
           rf.predict_proba(X_va_raw)[:, 1], sec)
else:
    print(f"건너뜀 - E01 기록값 사용: Valid Score {RF_SCORE}")
    RESULTS["A. RandomForest + 중앙값 대치"] = dict(
        train_brier=None, train_score=None,
        valid_brier=RF_BRIER, valid_score=RF_SCORE,
        fit_sec=None, extra={"note": "E01에서 측정. 이 노트북에서는 재실행하지 않음"})

## 5. LightGBM용 전처리

RandomForest와 다른 점 두 가지다.

**① 범주형을 정수 코드로 변환**

`train`에서 만든 고정 매핑을 저장해두고 추론 때 그대로 조회한다.
test를 보고 만들지 않으므로 규칙 위반이 아니다. (AGENTS.md §5)

**② 결측을 채우지 않는다**

LightGBM이 결측을 별도 분기로 처리한다.
`asof_*` 결측은 "데뷔 첫 투구"라는 의미 있는 정보이므로 보존한다.

In [ ]:
# train 기준으로 범주형 매핑 생성 (test를 보지 않음)
cat_maps = {}
for col in CAT_COLS:
    cats = sorted(train[col].dropna().unique().tolist())
    cat_maps[col] = {v: i for i, v in enumerate(cats)}
    print(f"{col:12s} {len(cats):2d}종  {cat_maps[col]}")

def encode(df):
    """행 단위 변환만 수행한다. 다른 행을 참조하지 않는다."""
    X = df[FEATURES].copy()
    for col in CAT_COLS:
        X[col] = X[col].map(cat_maps[col]).fillna(-1).astype("int32")
    return X

# (1) 결측 그대로 - LightGBM native 처리용
X_tr = encode(X_tr_raw)
X_va = encode(X_va_raw)

# (2) 중앙값 대치 - E01과 동일한 결측 처리 (효과 분리용)
#     중앙값은 반드시 학습 데이터에서만 계산해 검증에도 그대로 적용한다.
MEDIANS = X_tr.median(numeric_only=True)
X_tr_med = X_tr.fillna(MEDIANS)
X_va_med = X_va.fillna(MEDIANS)

print()
print("결측 컬럼 (LightGBM이 그대로 처리):")
na = X_tr.isna().sum()
print((na[na > 0] / len(X_tr) * 100).round(2).to_string())
print()
print(f"중앙값 대치 버전도 준비함 (X_tr_med / X_va_med) - 결측 {int(X_tr_med.isna().sum().sum())}개")

## 6. B0 - LightGBM 분류 + 중앙값 대치 (모델 효과만 분리)

E01(RandomForest)에서 **모델만** LightGBM으로 바꾼 조건이다.
결측 처리는 E01과 동일하게 중앙값 대치를 유지한다.

이 실험이 있어야 다음이 분리된다.

```
A  -> B0   :  모델 교체 효과      (RandomForest -> LightGBM)
B0 -> B    :  결측 처리 효과      (중앙값 대치 -> native)
```

두 변경을 한꺼번에 넣으면 어느 쪽이 기여했는지 알 수 없다. (AGENTS.md §15)

In [ ]:
PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=200,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=SEED,
    verbose=-1,
)

t = time.time()
clf_med = lgb.LGBMClassifier(**PARAMS)
clf_med.fit(
    X_tr_med, y_tr,
    eval_set=[(X_va_med, y_va)],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(100, verbose=False),
               lgb.log_evaluation(300)],
)
sec_b0 = time.time() - t
best_b0 = clf_med.best_iteration_

report("B0. LightGBM 분류 + 중앙값 대치",
       clf_med.predict_proba(X_tr_med)[:, 1],
       clf_med.predict_proba(X_va_med)[:, 1],
       sec_b0,
       extra={"사용 트리": f"{best_b0} / {PARAMS['n_estimators']}"})

## 7. B - LightGBM 분류 + 결측 native (logloss)

B0에서 **결측 처리만** 바꾼 조건이다. 결측을 채우지 않고 LightGBM이 직접 처리한다.

`asof_*` 결측은 "데뷔 첫 투구 / 첫 등판"이라는 정보다.
중앙값으로 채우면 이 신호가 사라진다.

In [ ]:
# PARAMS 는 B0 셀에서 정의한 것을 그대로 사용한다 (동일 조건 비교)
t = time.time()
clf = lgb.LGBMClassifier(**PARAMS)
clf.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(100, verbose=False),
               lgb.log_evaluation(300)],
)
sec_b = time.time() - t
best_b = clf.best_iteration_

report("B. LightGBM 분류 + 결측 native",
       clf.predict_proba(X_tr)[:, 1],
       clf.predict_proba(X_va)[:, 1],
       sec_b,
       extra={"사용 트리": f"{best_b} / {PARAMS['n_estimators']}"})

## 8. C - LightGBM 회귀 + 결측 native (L2 = Brier 직접 최적화)

B에서 **목적함수만** 바꾼 조건이다. 0/1 타깃을 그대로 회귀한다.
MSE가 곧 Brier Score이므로 평가 지표를 직접 최소화한다.

> "Best prediction != Optimal decision" - Decision-Focused Learning 3강

회귀는 0~1 범위를 벗어날 수 있으므로 `np.clip`으로 잘라준다.

In [ ]:
t = time.time()
reg = lgb.LGBMRegressor(objective="regression", **PARAMS)
reg.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="l2",
    callbacks=[lgb.early_stopping(100, verbose=False),
               lgb.log_evaluation(300)],
)
sec_c = time.time() - t
best_c = reg.best_iteration_

pred_tr_c = np.clip(reg.predict(X_tr), 0, 1)
pred_va_c = np.clip(reg.predict(X_va), 0, 1)

raw = reg.predict(X_va)
report("C. LightGBM 회귀 + 결측 native (L2=Brier)",
       pred_tr_c, pred_va_c, sec_c,
       extra={"사용 트리": f"{best_c} / {PARAMS['n_estimators']}",
              "clip 전 범위": f"{raw.min():.4f} ~ {raw.max():.4f}",
              "clip된 행": f"{int(((raw < 0) | (raw > 1)).sum()):,}"})

## 9. 결과 비교 - 변경 효과 분리

각 단계에서 **한 가지만** 바뀌므로 기여도를 분리할 수 있다.

```
A  RandomForest + 중앙값 대치
      | 모델 교체
B0 LightGBM 분류 + 중앙값 대치
      | 결측 처리
B  LightGBM 분류 + 결측 native
      | 목적함수
C  LightGBM 회귀 + 결측 native
```

In [ ]:
ORDER = [
    "A. RandomForest + 중앙값 대치",
    "B0. LightGBM 분류 + 중앙값 대치",
    "B. LightGBM 분류 + 결측 native",
    "C. LightGBM 회귀 + 결측 native (L2=Brier)",
]

rows = []
base = RESULTS[ORDER[0]]["valid_score"]
prev = None
for name in ORDER:
    if name not in RESULTS:
        continue
    r = RESULTS[name]
    rows.append({
        "실험": name,
        "Train Score": None if r["train_score"] is None else round(r["train_score"], 2),
        "Valid Score": round(r["valid_score"], 2),
        "Valid Brier": round(r["valid_brier"], 6),
        "vs A": round(r["valid_score"] - base, 2),
        "직전 대비": None if prev is None else round(r["valid_score"] - prev, 2),
        "학습(s)": None if r["fit_sec"] is None else round(r["fit_sec"], 1),
    })
    prev = r["valid_score"]

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

print("\n" + "=" * 60)
print("변경별 기여도 (직전 대비)")
print("=" * 60)
def delta(a, b):
    if a in RESULTS and b in RESULTS:
        return RESULTS[b]["valid_score"] - RESULTS[a]["valid_score"]
    return None

d_model  = delta(ORDER[0], ORDER[1])
d_na     = delta(ORDER[1], ORDER[2])
d_obj    = delta(ORDER[2], ORDER[3])
for label, d in [("모델 교체 (RF -> LGBM)", d_model),
                 ("결측 처리 (중앙값 -> native)", d_na),
                 ("목적함수 (logloss -> L2)", d_obj)]:
    if d is not None:
        print(f"  {label:32s} {d:+8.2f}")

best_name = max((n for n in ORDER if n in RESULTS),
                key=lambda k: RESULTS[k]["valid_score"])
print(f"\n최고: {best_name}  (Valid Score {RESULTS[best_name]['valid_score']:.2f})")

## 10. 결과 저장 및 EXPERIMENTS.md 기록용 출력

노트북 출력은 저장하지 않으면 사라지므로 **숫자를 파일로 남긴다.**
`results/e02_metrics.json` 에 환경 정보, 검증 설정, 전체 결과가 기록된다.
브랜치를 받은 사람이 노트북을 재실행하지 않아도 결과를 확인할 수 있다.

In [ ]:
from datetime import date

OWNER = "이준호"        # 필요하면 수정
TODAY = date.today().isoformat()

# (1) 결과를 파일로 저장
metrics_path = save_metrics("e02_metrics.json")

# (2) EXPERIMENTS.md 붙여넣기용 출력
print()
print("=" * 78)
print("EXPERIMENTS.md 요약표에 추가")
print("=" * 78)
LABEL = {
    "B0. LightGBM 분류 + 중앙값 대치":            ("E02a", "LightGBM 분류 (결측은 E01과 동일한 중앙값 대치)"),
    "B. LightGBM 분류 + 결측 native":             ("E02",  "LightGBM 분류 + 결측 native"),
    "C. LightGBM 회귀 + 결측 native (L2=Brier)":  ("E03",  "LightGBM 회귀(L2=Brier) + 결측 native"),
}
for name, (eid, label) in LABEL.items():
    if name not in RESULTS:
        continue
    r = RESULTS[name]
    mark = "채택" if r["valid_score"] > base else "보류"
    print(f"| {eid} | {TODAY} | {OWNER} | {label} | 2019~23 → 24 | "
          f"{r['valid_brier']:.6f} | {r['valid_score']:.2f} | "
          f"{r['valid_score'] - base:+.2f} | {mark} |")

print()
print("=" * 78)
print("상세 기록용")
print("=" * 78)
print(f"환경: lightgbm {ENV['lightgbm']} / scikit-learn {ENV['scikit-learn']} / "
      f"pandas {ENV['pandas']} / numpy {ENV['numpy']} / python {ENV['python']}")
print(f"검증: 2019~{VALID_SEASON-1} 학습({len(X_tr_raw):,}행) → {VALID_SEASON} 검증({len(X_va_raw):,}행)")
print()
for name in ORDER:
    if name not in RESULTS:
        continue
    r = RESULTS[name]
    print(f"--- {name} ---")
    if r["train_score"] is not None:
        print(f"  Train Brier {r['train_brier']:.6f}  Score {r['train_score']:.2f}")
    print(f"  Valid Brier {r['valid_brier']:.6f}  Score {r['valid_score']:.2f}")
    if r["fit_sec"]:
        print(f"  학습 시간 {r['fit_sec']:.1f}s")
    for k, v in r["extra"].items():
        print(f"  {k}: {v}")
    print()

## 11. 최종 모델 저장 (후보 전부)

세 조건(B0 / B / C) **모두** 전체 데이터(2019~2024)로 재학습해 각각 저장한다.

로컬 검증에서 세 조건의 차이가 몇 점 수준이라 우열을 판정할 수 없었다.
`Public Score = Private Score` 이므로 **리더보드가 가장 정확한 판정 기준**이다.
셋을 모두 제출 가능한 상태로 만들어 두고 실제 점수로 비교한다.

early stopping은 검증셋이 있어야 하므로, 재학습 시에는
검증에서 찾은 최적 트리 수에 데이터 증가분 10%를 더해 사용한다.

In [ ]:
# 후보 3개를 모두 전체 데이터로 재학습한다.
#   key -> (실험ID, RESULTS 이름, 예측방식, best_iteration, 결측처리)
CANDIDATES = {
    "b0": ("E02a", "B0. LightGBM 분류 + 중앙값 대치",            "proba", best_b0, "median"),
    "b":  ("E02",  "B. LightGBM 분류 + 결측 native",             "proba", best_b,  "native"),
    "c":  ("E03",  "C. LightGBM 회귀 + 결측 native (L2=Brier)",  "clip",  best_c,  "native"),
}

# 전체 데이터 (2019~2024)
X_all_native = encode(train[FEATURES])
y_all = train[TARGET]

# 중앙값은 전체 학습 데이터 기준으로 다시 계산한다 (test 를 보지 않음)
MEDIANS_ALL = X_all_native.median(numeric_only=True)
X_all_median = X_all_native.fillna(MEDIANS_ALL)

print(f"전체 학습 데이터: {len(X_all_native):,}행")
for key, (eid, name, kind, bi, na) in CANDIDATES.items():
    print(f"  {key:3s} {eid:5s} {kind:6s} 결측={na:7s} trees {bi} -> {int(bi*1.1)}")

In [ ]:
FINAL = {}

for key, (eid, name, kind, best_iter, na_mode) in CANDIDATES.items():
    final_n = int(best_iter * 1.1)
    fp = dict(PARAMS); fp["n_estimators"] = final_n
    X_use = X_all_median if na_mode == "median" else X_all_native

    t = time.time()
    if kind == "clip":
        model = lgb.LGBMRegressor(objective="regression", **fp)
    else:
        model = lgb.LGBMClassifier(**fp)
    model.fit(X_use, y_all)
    sec = time.time() - t

    artifact = {
        "features": FEATURES,
        "cat_cols": CAT_COLS,
        "cat_maps": cat_maps,
        "model":    model,
        "kind":     kind,
        # 중앙값 대치 조건이면 추론 때도 동일하게 채워야 한다.
        # 값은 train 에서 계산해 저장하므로 test 를 보지 않는다. (AGENTS.md §5, §6)
        "impute_median": (MEDIANS_ALL.to_dict() if na_mode == "median" else None),
        "meta": {
            "experiment":   eid,
            "selected":     name,
            "na_handling":  na_mode,
            "valid_score":  RESULTS[name]["valid_score"],
            "valid_brier":  RESULTS[name]["valid_brier"],
            "n_estimators": final_n,
            "params":       dict(fp),
            "environment":  ENV,
            "metrics_file": "results/e02_metrics.json",
        },
    }

    os.makedirs(MODEL_DIR, exist_ok=True)
    out = os.path.join(MODEL_DIR, f"lgbm_{key}.joblib")
    joblib.dump(artifact, out, compress=3)
    FINAL[key] = out
    print(f"[{key}] {eid} 재학습 {sec:5.1f}s -> {out} "
          f"({os.path.getsize(out)/1e6:.1f} MB)")

# 로컬 검증 최고 점수 모델을 기본 이름으로도 저장 (기존 경로 호환)
best_key = max(CANDIDATES, key=lambda k: RESULTS[CANDIDATES[k][1]]["valid_score"])
import shutil
shutil.copy(FINAL[best_key], os.path.join(MODEL_DIR, "lgbm.joblib"))
print(f"\n로컬 검증 최고: {best_key} ({CANDIDATES[best_key][0]}) -> models/lgbm.joblib 로도 복사")

## 12. 제출 파일 만들기

세 후보를 각각 제출 파일로 만든다. 터미널에서 실행한다.

```powershell
python scripts/make_submission.py --name e02a_lgbm_median --model models/lgbm_b0.joblib --lightgbm
python scripts/make_submission.py --name e02_lgbm_native  --model models/lgbm_b.joblib  --lightgbm
python scripts/make_submission.py --name e03_lgbm_l2      --model models/lgbm_c.joblib  --lightgbm
```

`make_submission.py` 는 artifact 에 기록된 lightgbm 버전을 읽어
`requirements.txt` 를 자동으로 같은 버전으로 맞춘다.

각 명령이 자동으로 수행하는 것:

1. `submissions/<이름>/` 구성 (model/, script.py, requirements.txt)
2. `script.py` 실제 실행 -> 작동 확인
3. `check_rules.py` 규칙 검사 (행 독립성 포함)
4. `submissions/<이름>.zip` 생성

### 왜 셋 다 제출하나

로컬 검증에서 세 조건의 차이가 3~4점 수준이라 우열을 판정할 수 없었다.
이 대회는 `Public Score = Private Score` (테스트 100%) 이므로
**리더보드가 가장 정확한 판정 기준**이다.

⚠️ 일일 제출 5회 제한을 고려할 것.

⚠️ 11번 셀의 재학습 모델은 2024를 학습에 포함하므로
2024로 채점하면 과대평가된다. 성능 추정치는 9번 셀의 비교표를 사용한다.